# STEP 4-A — 2단계 베이스라인

## 왜 한 번에 7클래스가 아니라 2단계인가

이 기능의 목적은 **"보호자에게 의심된다까지 알려주기"** 입니다.
그러면 가장 중요한 판단은 **"병원에 가봐야 하나?"** 이고, 그건 이진 문제입니다.
병변 종류를 6개로 나누는 건 그 다음 이야기죠.

A7(정상)까지 한 번에 7클래스로 풀면 두 문제가 섞입니다:

- A7(정상)을 A2(비듬)로 틀리는 것과 A2를 A3로 틀리는 것은
  **임상적 무게가 완전히 다른데**, 7클래스 손실함수는 둘을 똑같이 취급합니다.
- "놓치지 않는 게 우선" 이라는 **재현율 우선 임계값을 1단계에만** 걸 수가 없습니다.

그래서 나눕니다:

| 단계 | 하는 일 | 데이터 (VL01 기준) | 기준 |
|---|---|---|---|
| **1단계** | 정상(A7) vs 이상 | 22,815 : 23,070 — 거의 5:5 | **재현율 ≥ 0.95** |
| **2단계** | 병변 6종 (A1~A6) | 23,070장, A2 7,693 ↔ A5 1,464 = 5.3배 | macro-F1 |

## 이 노트북에서 하는 일

1. **크롭을 눈으로 확인** — 건너뛰면 안 되는 단계입니다
2. 1단계(정상/이상) 학습 → recall 0.95 지점의 임계값 확정
3. 2단계(병변 6종) 학습 → class weight 로 불균형 대응
4. **두 단계를 이어붙여** 사용자가 실제로 겪는 성능 측정 ★
5. 크롭 방식 비교 → 이후 실험에 쓸 크롭 확정

> ⚠️ 1단계와 2단계는 **같은 개체 단위 분할**을 씁니다.
> 단계별로 따로 나누면 1단계 검증에 쓴 강아지가 2단계 학습에 들어가 누수가 생깁니다.
> `stages.to_stage1/to_stage2` 가 `fold`/`is_holdout`/`group` 을 그대로 물려받습니다.

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
BRANCH = "main"
NAME   = "deeplearning_test"
_cwd   = os.getcwd()
if os.path.basename(_cwd) == NAME and os.path.isdir(os.path.join(_cwd, ".git")):
    DIR = _cwd            # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
else:
    BASE = "/content" if os.path.isdir("/content") else (
           "/kaggle/working" if os.path.isdir("/kaggle/working") else _cwd)
    DIR = os.path.join(BASE, NAME)

if os.path.isdir(os.path.join(DIR, ".git")):
    # 이미 받아둔 경우: 최신으로 강제 동기화 (shallow clone 에서도 안전)
    subprocess.run(["git", "-C", DIR, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO, DIR], check=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", subprocess.run(["git", "-C", DIR, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "timm", "imagehash", "pyarrow", "grad-cam"], check=False)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

## 0-b. 로컬에서 만든 데이터 불러오기

한국 PC 에서 `prepare_local.py` 로 전처리한 `dogskin_prepared.zip` 을 가져옵니다.

> 🚨 **AI Hub 는 해외 IP 다운로드를 차단**해서 Colab 에서는 원본을 받을 수 없습니다.
> 다운로드·전처리는 로컬에서, 학습만 여기서 합니다.
> → [`docs/cautions/06`](../docs/cautions/06_해외IP_다운로드_차단_우회.md)

**준비**: `dogskin_prepared.zip` 을 Google Drive 에 올려두세요 (Kaggle 이면 Add Input).

In [ ]:
# Colab 이면 Drive 마운트
if E.env == "colab":
    env.mount_drive()

# zip 을 찾아 로컬 디스크로 풉니다 (Drive 에서 직접 읽으면 학습이 10배 느립니다)
env.load_prepared()          # 경로를 직접 주려면: env.load_prepared("/content/drive/MyDrive/dogskin_prepared.zip")

In [ ]:
import torch
from src import labels, split, crop, data, models, train, evaluate, stages
from src.config import CLASSES_STAGE1, NORMAL_LABEL

DEV = "cuda" if torch.cuda.is_available() else "cpu"

df = labels.load(env.work_root()/"manifests"/"manifest_final.parquet")
print(f"{len(df):,}행 / 개체 {df['animal_id'].nunique():,}마리")
print("클래스 분포:", df["label"].value_counts().to_dict())
print("사용 가능한 크롭 태그:", crop.available_tags())

# 로컬에서 개체 단위 분할까지 끝냈으므로 fold/holdout 컬럼이 들어 있습니다
split.verify(df, fold=0, strict=True)

## 1. 크롭 검증 — 숫자로 먼저, 눈으로는 딱 하나만

여기서 확인해야 하는 건 "병변이 맞는가" 가 **아닙니다.** 그건 수의사의 일이고,
저 데이터의 라벨은 이미 수의사가 붙인 것입니다. 우리가 확인할 건 다른 겁니다:

> **크롭이 라벨을 다른 경로로 흘리고 있지 않은가.**

무슨 뜻이냐면 — A7(정상)에도 라벨러가 "촬영한 피부 부위" 박스를 찍어놨습니다.
그래서 A7 도 `area_ratio` 값이 있습니다. 그 자체는 정상인데, 만약
**정상 박스가 병변 박스보다 계통적으로 크다면** 크롭의 확대 배율만 봐도
정상/병변이 티가 납니다. 그러면 모델은 피부를 안 보고 **줌 레벨을 셉니다.**

이런 지름길(shortcut)은 검증 점수를 **높게** 만듭니다. 그래서 숫자로 잡아야 합니다.
`crop.audit()` 이 그걸 포함해 5가지를 재줍니다 — 의학 지식이 필요 없습니다.

In [ ]:
BEST_CROP = "m1.5"                       # 아래 6번에서 비교 후 확정합니다
IMG_SIZE  = 224                          # 아래 학습에서 쓸 입력 크기와 맞춰야 의미가 있습니다
d = crop.switch_tag(df, BEST_CROP)

report = crop.audit(d, cfg=CFG(img_size=IMG_SIZE))

### 🚦 감사 결과 읽기

| 결과 | 의미 | 대응 |
|---|---|---|
| `[1](a)` 정상/병변 배율 **1.5배 이상** | 크롭 배율이 정상/이상을 흘림 | **1단계를 `full` 크롭으로** (자동) |
| `[1](b)` 병변 6종 간 배율 **2배 이상** | 크롭 배율이 병변 종류를 흘림 | 고정 픽셀 크롭 필요 (아래 판단) |
| `[2]` 확대 비율이 50% 넘음 | 없는 디테일을 만들어 냄 | `[1](b)` 와 같은 원인 |
| `[3]` 흐림 비율 30% 넘음 | 원본 사진 품질 한계 | 실사용에서 흐린 사진 거절 (05) |
| `[3]` 정상/병변 선명도 격차 | 배율 차이의 **부작용** | `[1]` 을 고치면 함께 완화됨 |
| `[4]` 라벨 충돌 > 0 | 같은 파일명에 다른 라벨 | 멈추고 알려주세요 |
| `[5]` 이탈량 50px 미만 | 라벨러 오차 | 무시 가능 (크롭이 알아서 잘라 넣음) |
| `[5]` 이탈량 50px 이상 | 좌표 해석 오류 | 멈추고 알려주세요 |

`[1]`과 `[3]`은 **같은 원인**입니다: 박스가 작으면 → 더 확대되고 → 흐려집니다.
따로 고칠 문제가 아닙니다.

### `full` 크롭의 대가 — 병변이 화면 밖으로 나가는 비율

배율 지름길을 막는 가장 확실한 방법은 박스를 아예 안 쓰는 `full`(중앙 정사각) 크롭입니다.
그런데 **병변이 좌우 끝에 있으면 화면에서 잘려 나갑니다.**
그 사진은 "이상"인데 정상처럼 보이니, **1단계 recall의 천장이 데이터 때문에 낮아집니다.**

목표가 recall 0.95인데 천장이 0.90이면 `full`은 못 씁니다. 미리 재고 정합니다.

In [ ]:
loss_full = crop.full_crop_loss(df, "full", cfg=CFG(img_size=IMG_SIZE))

### 얼마나 심각한가 — 사진을 안 보고 맞혀보기 ★

배율 차이가 있다는 건 알았습니다. 그런데 **그게 실제로 얼마나 정답을 흘리는지**는
따로 재야 합니다. 방법은 간단합니다: **픽셀을 한 장도 안 보고** 박스 크기·모양만으로
분류기를 학습시켜 봅니다.

거기서 나오는 점수가 CNN 이 넘어야 하는 **하한선**입니다.

```
CNN macro-F1 0.45  vs  메타데이터만 0.40   →  피부에서 얻은 건 0.05 뿐  🚨
CNN macro-F1 0.45  vs  메타데이터만 0.18   →  대부분 피부에서 얻음     ✅
```

같은 `fold` 를 쓰므로 4번의 CNN 점수와 직접 비교할 수 있습니다.

In [ ]:
# ★ 판단 기준: 크롭 배율로 **이미지에 실제로 보이는** 특징만 씁니다.
floor = crop.shortcut_baseline(d, cfg=CFG(img_size=IMG_SIZE), features="scale_only")

# 참고: 메타데이터 전체를 넣으면 얼마나 나오는지 (종횡비·병변개수·해상도 포함).
# 그것들은 크롭에 안 보이므로 f320 판단에 쓰면 안 됩니다 — 데이터 성질 참고용.
floor_all = crop.shortcut_baseline(d, cfg=CFG(img_size=IMG_SIZE), features="all",
                                   verbose=False)
print(f"\n(참고) 메타데이터 전체 기준: 1단계 AUROC "
      f"{floor_all.get('stage1_auroc_metadata_only', float('nan')):.4f}, "
      f"2단계 macro-F1 {floor_all.get('stage2_macro_f1_metadata_only', float('nan')):.4f}")
print("  이 값이 위보다 높다면, 크롭에 안 보이는 정보(병변 개수 등)가 라벨과")
print("  상관이 있다는 뜻입니다. CNN 은 그걸 못 쓰므로 판단에는 위 숫자를 쓰세요.")

### 판단: 크롭을 다시 만들어야 하나?

위 하한선을 보고 정합니다.

| 하한선 (2단계 macro-F1) | 판단 | 할 일 |
|---|---|---|
| **< 0.30** | 배율 지름길이 약함 | 그냥 진행. 4번에서 CNN 이 하한선을 넘는지 확인 |
| **≥ 0.30** | 배율이 정답을 크게 흘림 | **고정 픽셀 크롭을 만드세요** ↓ |

고정 픽셀 크롭(`f320`)은 병변 중심에서 **항상 320px**을 잘라냅니다.
피부 1mm 가 항상 같은 픽셀 수라 배율로 맞히는 경로가 막힙니다.
대신 큰 병변은 창을 넘어 잘립니다 — 그게 대가입니다.

**로컬 PC 에서** (원본이 있어야 합니다):

```cmd
py prepare_local.py --chunk VL01 --margins -320
py prepare_local.py --finalize
py prepare_local.py --package
```

기존 크롭은 그대로 두고 `f320` 태그만 추가되므로, 올린 뒤 `crop.available_tags()` 에
`f320` 이 보이면 6번의 크롭 비교에 자동으로 포함됩니다.

> 💡 하한선이 애매하면(0.25~0.30) 일단 진행하세요. 4번에서 CNN 점수와 비교한 뒤
> 격차가 작으면 그때 다시 만들면 됩니다. 지금 확실히 아는 건 하한선뿐입니다.

### 눈으로 볼 것 — 딱 하나

아래 표는 클래스마다 한 줄씩입니다. 병변인지 아닌지 판단하지 마세요.
**줄끼리 서로 달라 보이는지만** 보세요.

- 줄마다 달라 보인다 → 모델이 배울 신호가 있습니다 ✅
- 전부 똑같은 털 사진처럼 보인다 → 모델도 구분 못 할 가능성이 큽니다 ⚠️
  (그렇다고 실패는 아닙니다. 모델은 사람이 못 보는 질감 차이를 봅니다.
   다만 기대치를 낮추고, 4번의 macro-F1 을 냉정하게 보셔야 합니다)
- 한 줄 안에서 제각각이다 → 라벨이 섞였을 수 있습니다
- 개 피부/털이 아니라 사람 손·바닥만 보인다 → 크롭이 어긋난 것

In [ ]:
crop.contact_sheet(d, per_class=6)

### 참고: 박스가 어디에 얹혔는지

좌표계가 뒤집혔거나(x↔y) 스케일이 어긋났으면 여기서 드러납니다.
박스가 **크롭 가운데를 크게 차지하는 게 정상**입니다 (margin 1.5 → 폭의 약 2/3).
박스가 구석에 처박혀 있거나 화면을 벗어나면 좌표 해석이 틀린 겁니다.

In [ ]:
crop.preview_with_box(d, n=4)

## 2. 2단계 뷰 만들기

같은 매니페스트에서 **두 개의 뷰**를 만듭니다. 데이터를 복사하는 게 아니라
`label` 컬럼만 다르게 보는 겁니다.

```
df  ──▶ to_stage1()  label: A7 / ABNORMAL      전체 45,885행
    └─▶ to_stage2()  label: A1~A6              병변 23,070행만
```

`fold` / `is_holdout` / `group` 은 그대로 따라옵니다. → 두 단계가 **같은 분할**을 씁니다.

> ⚠️ **두 단계가 서로 다른 크롭을 쓸 수 있습니다.** 위 감사에서 정상/병변의 박스
> 배율이 다르게 나왔다면, 1단계는 `full` 크롭을 씁니다 — ROI 크롭이 배율로
> 정답을 흘리기 때문입니다. 분할은 여전히 공유하므로 누수는 생기지 않습니다.

In [ ]:
# ★ 1단계 크롭 결정 — 두 가지를 함께 봅니다.
#   ① 배율 격차: ROI 크롭은 그 배율 자체를 정답으로 흘립니다
#   ② full 의 천장: full 은 박스를 안 쓰지만 병변을 화면 밖으로 잃습니다
TARGET  = CFG().target_recall_stage1          # 보통 0.95
SCALE_GAP = report.get("area_ratio_normal_over_lesion", 1.0)
CEILING   = loss_full.get("stage1_recall_ceiling", 1.0)
TAGS      = crop.available_tags()
FIXED     = next((t for t in TAGS if t.startswith("f") and t[1:].isdigit()), None)

leaky = SCALE_GAP > 1.5 or SCALE_GAP < 0.67

if not leaky:
    STAGE1_CROP, why = BEST_CROP, "배율 지름길이 약해 ROI 크롭을 그대로 씀"
elif FIXED:
    # 고정 픽셀 크롭이 최선입니다 — 배율이 일정하면서 병변도 항상 화면에 들어옵니다
    STAGE1_CROP = FIXED
    why = f"배율 격차 {SCALE_GAP:.2f}배 → 고정 픽셀 크롭 '{FIXED}' (배율 일정 + 병변 보존)"
elif CEILING >= TARGET + 0.01:
    STAGE1_CROP = "full"
    why = (f"배율 격차 {SCALE_GAP:.2f}배 → full (박스 미사용). "
           f"천장 {CEILING:.3f} 이 목표 {TARGET:.2f} 보다 높아 사용 가능")
else:
    STAGE1_CROP = BEST_CROP
    why = (f"배율 격차 {SCALE_GAP:.2f}배지만 full 천장 {CEILING:.3f} 이 "
           f"목표 {TARGET:.2f} 에 못 미치고, 고정 픽셀 크롭도 없음")

print(f"1단계 크롭: '{STAGE1_CROP}'\n  근거: {why}")
if STAGE1_CROP == "full":
    print(f"  ⚠️ 천장 {CEILING:.3f} — 여유가 {CEILING - TARGET:.3f} 뿐입니다.")
    print("     1단계 recall 이 목표에 못 미치면 모델 문제가 아니라 이 천장 때문일 수 있습니다.")
    print("     f320 을 만들면 배율도 잡고 병변도 안 잃습니다 (--margins -320).")
elif leaky and STAGE1_CROP == BEST_CROP:
    print("\n  🚨 지름길을 막지 못한 상태로 진행합니다. 점수를 낙관적으로 취급하세요.")
    print("     · 로컬에서: py prepare_local.py --chunk VL01 --margins -320")
    print("     · 6번 배율 교란 검사로 실제 의존도를 확인하세요")

# 2단계(병변 종류)는 ROI 크롭을 씁니다 — 형태를 구분하려면 병변을 크게 봐야 합니다.
s1_all = stages.to_stage1(crop.switch_tag(df, STAGE1_CROP))
s2_all = stages.to_stage2(d)

# 두 뷰 각각 누수 재확인 — 뷰를 만드는 과정에서 분할이 깨지지 않았는지
split.verify(s1_all, fold=0, strict=True)
split.verify(s2_all, fold=0, strict=True)

## 3. 1단계 — 정상 / 이상

거의 5:5 라 학습이 수월합니다. 대신 **평가 기준이 다릅니다**:
macro-F1 이 아니라 **재현율(recall)** 이 먼저입니다.

> 오탐(정상인데 병원 가보라고 함) = 보호자가 헛걸음
> 미탐(병변인데 괜찮다고 함) = **놓친 병**
>
> 둘의 비용이 전혀 다르므로 recall 을 0.95로 **먼저 고정**하고,
> 그 조건에서 precision 이 얼마나 나오는지를 봅니다.

📖 [`docs/basics/08_확률보정과_임계값_결정.md`](../docs/basics/08_확률보정과_임계값_결정.md)

In [ ]:
cfg1 = CFG(
    model_name="resnet50",
    img_size=224,
    epochs=8,
    balance_strategy="none",         # 5:5 라 가중치 불필요
    monitor="macro_f1",
    exp_name=f"stage1_resnet50_{STAGE1_CROP}",
)
tr1, va1 = split.get_fold(s1_all, cfg1.use_fold)
print(f"1단계  train {len(tr1):,} / val {len(va1):,}   배치={cfg1.resolved_batch_size()}")

m1 = models.build("resnet50", n_classes=len(CLASSES_STAGE1),
                  pretrained=True, drop_rate=cfg1.drop_rate)
dl_tr1, dl_va1, ds_tr1, _ = data.build_loaders(tr1, va1, cfg1, model=m1,
                                               classes=CLASSES_STAGE1)

In [ ]:
res1 = train.fit(m1, dl_tr1, dl_va1, cfg1, ds_train=ds_tr1)
res1.plot()

In [ ]:
# 검증셋 점수 → '이상일 확률' → recall 0.95 지점의 임계값
_, lg1_va, y1_va = train.evaluate_loader(m1, dl_va1, None, DEV,
                                         len(CLASSES_STAGE1), tta_hflip=cfg1.tta_hflip)
scores1 = stages.stage1_scores(lg1_va)
ybin1 = stages.binary_targets(y1_va)

bin1 = evaluate.binary_report(scores1, ybin1, target_recall=cfg1.target_recall_stage1)
THR1 = bin1["threshold"]

### 🚦 1단계 게이트

- **AUROC ≥ 0.80** 이어야 이진 판정이 의미가 있습니다. 0.5는 동전 던지기입니다.
- recall 0.95 조건에서 precision 이 **0.5 미만**이면 오탐이 절반 넘습니다.
  → 쓸 수는 있지만("의심되니 가보세요" 니까) 사용자 신뢰가 빨리 깎입니다. 개선 대상입니다.

In [ ]:
assert bin1["auroc"] > 0.80, (
    f"1단계 AUROC {bin1['auroc']:.3f} — 정상/이상을 거의 구분하지 못합니다.\n"
    "모델을 바꾸기 전에 크롭과 라벨을 다시 확인하세요 (위 1번 셀)."
)
print(f"✅ 1단계 통과 — AUROC {bin1['auroc']:.4f}, 임계값 {THR1:.4f}")
if bin1["precision_at_target"] < 0.5:
    print(f"⚠️ 다만 precision {bin1['precision_at_target']:.3f} — 알림 절반 이상이 헛알림입니다.")

# ★ 하한선과 비교
base1 = floor.get("stage1_auroc_metadata_only")
if base1 is not None:
    print(f"\n  사진을 안 본 하한선 AUROC : {base1:.4f}")
    print(f"  CNN AUROC                 : {bin1['auroc']:.4f}")
    if bin1["auroc"] - base1 < 0.05:
        print("  🚨 하한선을 거의 못 넘었습니다 — 배율만 보고 있을 수 있습니다.")
        print(f"     (1단계 크롭은 '{STAGE1_CROP}' 입니다. full 인데도 이러면 다른 원인입니다)")
    else:
        print("  ✅ 하한선을 넘었습니다.")

## 4. 2단계 — 병변 6종

여기가 어려운 쪽입니다. 불균형이 **5.3배**(A2 7,693 ↔ A5 1,464)이고,
병변 형태끼리 실제로 닮았습니다.

- `balance_strategy="class_weight"` → 희소 클래스에 손실 가중치 (1/√n)
- 주 지표는 **macro-F1** — 클래스마다 같은 무게를 줍니다
- ⚠️ 색상 증강을 약하게 잡아둔 이유: **피부 병변은 색이 곧 라벨**입니다.
  A3(과다색소침착)의 어두운 색을 밝게 만들면 A1이 됩니다. 증강이 라벨을 파괴하죠.

📖 [`docs/basics/06_과적합_정규화_데이터증강.md`](../docs/basics/06_과적합_정규화_데이터증강.md)

In [ ]:
cfg2 = CFG(
    model_name="resnet50",
    img_size=224,
    epochs=12,
    balance_strategy="class_weight",     # 5.3배 불균형
    monitor="macro_f1",
    exp_name=f"stage2_resnet50_{BEST_CROP}",
)
tr2, va2 = split.get_fold(s2_all, cfg2.use_fold)
print(f"2단계  train {len(tr2):,} / val {len(va2):,}")

m2 = models.build("resnet50", n_classes=len(CLASSES),
                  pretrained=True, drop_rate=cfg2.drop_rate)
dl_tr2, dl_va2, ds_tr2, _ = data.build_loaders(tr2, va2, cfg2, model=m2, classes=CLASSES)

In [ ]:
# 증강이 실제로 뭘 하는지 눈으로 보기
import matplotlib.pyplot as plt
from src.data import IMAGENET_MEAN, IMAGENET_STD

x, y = next(iter(dl_tr2))
mean = torch.tensor(IMAGENET_MEAN).view(3,1,1); std = torch.tensor(IMAGENET_STD).view(3,1,1)
fig, axes = plt.subplots(2, 4, figsize=(12, 6.2))
for ax, i in zip(axes.flat, range(min(8, len(x)))):
    ax.imshow((x[i]*std+mean).clamp(0,1).permute(1,2,0)); ax.axis("off")
    ax.set_title(f"{CLASSES[y[i]]} {CLASS_KO[CLASSES[y[i]]][:8]}", fontsize=8)
plt.suptitle("증강 후 실제로 모델이 보는 이미지"); plt.tight_layout(); plt.show()
print("💡 병변이 잘려 나가거나 색이 심하게 변했다면 증강이 너무 센 겁니다.")

In [ ]:
res2 = train.fit(m2, dl_tr2, dl_va2, cfg2, ds_train=ds_tr2)
res2.plot()

### 학습 곡선 읽는 법

| 증상 | 의미 | 대응 |
|---|---|---|
| train↓ val↓ 둘 다 계속 하락 | 정상, 더 학습 가능 | epochs 늘리기 |
| train↓ **val↑** | 과적합 시작 | 조기종료 지점, 증강↑ / drop_rate↑ |
| 둘 다 안 내려감 | 학습이 안 됨 | lr 조정, 데이터/라벨 확인 |
| val 이 심하게 출렁임 | 배치가 작거나 lr 이 큼 | batch↑ 또는 lr↓ |

In [ ]:
_, lg2_va, y2_va = train.evaluate_loader(m2, dl_va2, None, DEV,
                                         len(CLASSES), tta_hflip=cfg2.tta_hflip)
rep2 = evaluate.full_report(lg2_va, y2_va, CLASSES)
rep2.plot_confusion()
rep2.plot_per_class()

### 🚦 2단계 게이트

**macro-F1 이 0.25 미만이면 여기서 멈추고 데이터를 다시 보세요.**
랜덤이 1/6 ≈ 0.167 인데 그것보다 조금 나은 수준이면 파이프라인 어딘가가 깨진 겁니다.

흔한 원인: 라벨 매칭 오류, 크롭 좌표 오류, 클래스 매핑 뒤바뀜.

In [ ]:
assert rep2.metrics["macro_f1"] > 0.25, (
    "2단계가 랜덤 수준입니다. 모델을 바꾸지 말고 크롭·라벨을 다시 확인하세요."
)
cnn_f1 = rep2.metrics["macro_f1"]
print(f"✅ 2단계 게이트 통과 — macro-F1 {cnn_f1:.4f}")

# ── 하한선과 비교 (전체) ────────────────────────────────────────
base = floor.get("stage2_macro_f1_metadata_only")
if base is not None:
    lift = cnn_f1 - base
    print(f"\n  사진을 안 본 하한선 : {base:.4f}")
    print(f"  CNN                : {cnn_f1:.4f}")
    print(f"  피부에서 얻은 것   : {lift:+.4f}")
    if lift < 0.05:
        print("\n  🚨 CNN 이 하한선을 거의 못 넘었습니다 — 크롭 배율을 보고 있을 수 있습니다.")
    elif lift < 0.15:
        print("\n  ⚠️ 격차가 작습니다. 배율 정보에 상당히 의존하고 있습니다.")
    else:
        print("\n  ✅ CNN 이 하한선을 크게 넘었습니다.")

# ── 클래스별 비교 ───────────────────────────────────────────────
# ⚠️ 차이의 **부호**가 뜻이 정반대입니다:
#     차이 > 0 인데 작다  → 크기 정보에 얹혀 있을 수 있음 (지름길 의심)
#     차이 < 0            → 크기보다도 못함 = 크기를 **안** 쓰는 중.
#                           지름길이 아니라 그 병변 자체를 못 잡는 문제입니다.
#   둘을 같은 경고로 묶으면 엉뚱한 대응(재크롭)을 하게 됩니다.
base_rec = floor.get("stage2_recall_metadata_only") or {}
if base_rec:
    cnn_rec = rep2.metrics["per_class"]["recall"]
    print(f"\n  {'클래스':<8}{'하한선':>9}{'CNN':>9}{'차이':>9}   판정")
    leaky, weak = [], []
    for i, c in enumerate(CLASSES):
        b, v = base_rec.get(c, 0.0), cnn_rec[i]
        d = v - b
        if b > 0.3 and 0 <= d < 0.15:
            note, _ = "⚠️ 크기에 얹혀 있을 수 있음", leaky.append(c)
        elif d < 0:
            note, _ = "→ 크기를 안 씀. 이 병변이 어려운 것", weak.append(c)
        else:
            note = ""
        print(f"  {c:<8}{b:>9.3f}{v:>9.3f}{d:>+9.3f}   {note}")

    if leaky:
        print(f"\n  ⚠️ 지름길 의심: {', '.join(leaky)}")
        print("     6번 배율 교란 검사와 05 의 Grad-CAM 으로 확인하세요.")
    if weak:
        print(f"\n  📉 크기보다도 못한 클래스: {', '.join(weak)}")
        print("     이건 지름길 문제가 **아닙니다.** 크기를 단서로 썼다면 최소한")
        print("     하한선만큼은 나왔을 테니까요. 재크롭으로 해결되지 않습니다.")
        print("     → 원인은 학습 부족 / 표본 부족 / 그 병변의 난이도입니다.")
        if "A6" in weak:
            print("     → A6(결절·종괴)는 종양 감별 클래스입니다. recall 을 올리는 게")
            print("        이 프로젝트에서 가장 중요한 개선 과제입니다.")
    if not leaky and not weak:
        print("\n  ✅ 모든 클래스가 하한선을 유의미하게 넘었습니다.")

# ── 수렴 여부 — 에폭을 더 줘야 하는지 ───────────────────────────
h = res2.history
if len(h) >= 3:
    last3 = [r["val_macro_f1"] for r in h[-3:]]
    best_at_end = res2.best_epoch >= len(h) - 2
    still_rising = last3[-1] >= max(last3[:-1])
    if best_at_end and still_rising:
        print(f"\n  📈 마지막 에폭({res2.best_epoch})이 최고입니다 — 아직 수렴하지 않았습니다.")
        print(f"     val loss 도 계속 내려가는 중이면 epochs 를 2배로 늘려보세요.")
        print(f"     (지금 {cfg2.epochs} → 25~30). 과적합이 아니라 학습 부족입니다.")

## 5. 두 단계를 이어붙이기 ★ 여기가 진짜 성능

각 단계를 따로 잘 하는 것과, **이어붙여서** 잘 하는 것은 다릅니다.
**1단계가 놓친 병변은 2단계가 볼 기회조차 없습니다.**
사용자가 실제로 겪는 건 이 이어붙인 결과입니다.

```
사진 → 1단계 ─ '이상 확률' < 임계값 → "정상으로 보입니다"   (2단계는 안 봄)
              └ 임계값 이상 ────────→ 2단계 → "A2 소견이 의심됩니다"
```

⚠️ 평가할 때 중요한 점: **2단계 모델도 정상 사진에 돌려야 합니다.**
실제 서비스에서는 정상 사진도 1단계를 통과하면 2단계로 넘어오니까요.
그래서 두 모델을 **같은 검증셋(정상 포함), 같은 순서**로 돌립니다.

In [ ]:
# 전체 검증셋 = 정상 + 병변. 두 모델을 같은 행·같은 순서로 돌립니다.
va_all = split.get_fold(s1_all, cfg1.use_fold)[1]        # 1단계 뷰 (label_orig 보존)

# ⚠️ 두 단계가 다른 크롭을 쓸 수 있습니다 (1단계 full / 2단계 m1.5).
#    각 모델에는 **그 모델이 학습한 크롭**을 먹여야 합니다.
#    switch_tag 는 image_path 해시로 경로를 다시 계산하므로 행 순서가 보존됩니다.
va_s1 = va_all
va_s2 = crop.switch_tag(va_all, BEST_CROP, verbose=False) if STAGE1_CROP != BEST_CROP \
        else va_all
print(f"1단계 입력 크롭: {STAGE1_CROP}  |  2단계 입력 크롭: {BEST_CROP}")

dl_e1, ds_e1 = data.eval_loader(va_s1, cfg1, model=m1, classes=CLASSES_STAGE1)
dl_e2, ds_e2 = data.eval_loader(va_s2, cfg2, model=m2, classes=CLASSES)

# 순서가 어긋나면 점수가 조용히 엉망이 됩니다 — 반드시 확인
assert len(ds_e1.df) == len(ds_e2.df), \
    f"행 수가 다릅니다: {len(ds_e1.df)} vs {len(ds_e2.df)} — 한쪽 크롭이 빠졌습니다"
assert (ds_e1.df["image_name"].to_numpy() == ds_e2.df["image_name"].to_numpy()).all(), \
    "두 로더의 행 순서가 다릅니다"

_, lg1_e, _ = train.evaluate_loader(m1, dl_e1, None, DEV, len(CLASSES_STAGE1), tta_hflip=True)
_, lg2_e, _ = train.evaluate_loader(m2, dl_e2, None, DEV, len(CLASSES), tta_hflip=True)

s1_sc = stages.stage1_scores(lg1_e)
y_final = ds_e1.df["label_orig"].to_numpy()               # A1~A7 원래 라벨
print(f"평가 대상 {len(y_final):,}장 (정상 {(y_final == NORMAL_LABEL).sum():,} / "
      f"병변 {(y_final != NORMAL_LABEL).sum():,})")

In [ ]:
pipe = stages.pipeline_report(s1_sc, lg2_e, y_final, threshold=THR1)
stages.plot_pipeline_confusion(pipe)

## 6. 실사용 견고성 검사 ★ 여기서 진짜가 드러납니다

지금까지의 점수는 모두 **우리가 만든 크롭** 위에서 잰 것입니다.
그 크롭은 병변을 정중앙에 두고, 병변 크기에 맞춰 배율을 정했습니다.
보호자 사진은 둘 다 아닙니다.

그래서 검증셋을 일부러 그렇게 망가뜨려 보고 점수 하락폭을 잽니다.
**하락폭이 곧 실사용 위험도**입니다. 학습은 안 하니 몇 분이면 됩니다.

| 하락폭 | 판정 |
|---|---|
| 15% 미만 | ✅ 견고 |
| 15~30% | ⚠️ 상당히 의존 — 개선 여지 큼 |
| 30% 이상 | 🚨 실사용에서 무너짐 |

> 💡 **`f320`의 효과는 하한선이 아니라 이 숫자로 판정하세요.**
> 하한선(`shortcut_baseline`)은 `bbox` 컬럼을 쓰기 때문에 크롭 방식을 바꿔도
> 거의 안 변합니다. "데이터에 상관이 있나"와 "모델이 그걸 썼나"는 다른 질문입니다.

In [ ]:
from src import robust

rb = robust.report(m2, va2, cfg2, CLASSES, n=2000)

### 임계값을 바꾸면 무엇이 바뀌나

임계값 하나가 이 시스템의 **성격**을 정합니다.
낮추면 놓치는 병변은 줄고 헛알림이 늘어납니다.

이 프로젝트의 목적("의심된다까지 알려주기")에서는 **놓치지 않는 쪽**이 맞습니다.
다만 헛알림이 너무 많으면 보호자가 알림을 무시하게 되므로, 표를 보고 균형점을 잡으세요.

In [ ]:
import pandas as pd
import numpy as np

rows = []
for t in np.quantile(s1_sc, [0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65]):
    r = stages.pipeline_report(s1_sc, lg2_e, y_final, threshold=float(t), show=False)
    rows.append({"임계값": round(float(t), 4),
                 "놓친병변": r["lesion_missed"],
                 "스크리닝recall": round(r["lesion_screening_recall"], 4),
                 "헛알림비율": round(r["false_alarm_rate"], 4),
                 "종류정확도": round(r["kind_accuracy_given_routed"], 4),
                 "최종macroF1": round(r["final_macro_f1"], 4)})
tbl = pd.DataFrame(rows)
print(tbl.to_string(index=False))
print(f"\n선택한 임계값: {THR1:.4f} (recall {cfg1.target_recall_stage1:.0%} 목표 기준)")
print("💡 '스크리닝recall' 이 0.95를 넘는 가장 큰 임계값을 고르면 헛알림이 최소가 됩니다.")

## 7. 증강으로 배율 의존을 줄여보기 (재크롭 없이)

위에서 하락폭이 컸다면, **재크롭 전에 이걸 먼저 시도하세요.** 설정만 바꾸므로
로컬 작업도 재업로드도 필요 없습니다.

원리: 학습 때 배율을 크게 흔들어 놓으면 클래스별 배율 분포가 서로 겹쳐서
"배율로 맞히기"가 통하지 않게 됩니다. 그러면 모델이 다른 걸 봐야 합니다.

```
기본       rrc_scale=(0.70, 1.0)   면적 1.4배 = 선형 1.2배   ← 막아야 할 격차가 2.5배인데 무의미
scale_robust      (0.35, 1.0)   면적 2.9배 = 선형 1.7배
scale_robust_hard (0.18, 1.0)   면적 5.6배 = 선형 2.4배
```

⚠️ **여유 있는 크롭이 필요합니다.** `m1.5`처럼 딱 붙은 크롭에 강한 확대 증강을 걸면
병변이 화면에서 잘려 나가 라벨이 깨집니다. `m2.5`가 있으면 그걸 base로 쓰세요.

In [ ]:
import gc

from src.config import with_aug

AUG_BASE_CROP = "m2.5" if "m2.5" in crop.available_tags() else BEST_CROP
print(f"증강 실험 base 크롭: {AUG_BASE_CROP} (여유가 있어야 강한 확대 증강이 안전합니다)")

s2_aug = stages.to_stage2(crop.switch_tag(df, AUG_BASE_CROP), verbose=False)
ta, va_a = split.get_fold(s2_aug, 0)

aug_results, aug_robust = {}, {}
for preset in ("default", "scale_robust"):
    cfg_a = with_aug(CFG(model_name="resnet50", img_size=IMG_SIZE, epochs=10,
                         balance_strategy="class_weight", monitor="macro_f1",
                         exp_name=f"s2_aug_{AUG_BASE_CROP}"), preset)
    print(f"\n{'='*60}\n  증강: {preset}  rrc_scale={cfg_a.rrc_scale}\n{'='*60}")
    ma = models.build("resnet50", len(CLASSES), pretrained=True, verbose=False)
    la, lv, da, _ = data.build_loaders(ta, va_a, cfg_a, model=ma, classes=CLASSES)
    train.fit(ma, la, lv, cfg_a, ds_train=da)
    _, lg, yy = train.evaluate_loader(ma, lv, None, DEV, len(CLASSES), tta_hflip=True)
    aug_results[preset] = evaluate.full_report(lg, yy, CLASSES, show=False)
    # ★ 검증 점수만 보면 안 됩니다 — 견고성이 목적입니다
    aug_robust[preset] = robust.scale_stress(ma, va_a, cfg_a, CLASSES, n=1500)
    del ma; gc.collect()
    if DEV == "cuda":
        torch.cuda.empty_cache()

evaluate.compare_models(aug_results)

In [ ]:
print(f"{'증강':<20}{'macro-F1':>10}{'배율하락':>12}")
for k in aug_results:
    f1 = aug_results[k].metrics["macro_f1"]
    dr = aug_robust[k].get("_summary", {}).get("rel_drop", float("nan"))
    print(f"{k:<20}{f1:>10.4f}{dr:>11.1%}")

print("\n읽는 법:")
print("  · macro-F1 이 조금 떨어지고 배율하락이 크게 줄었다  → ✅ 증강이 성공. 재크롭 불필요")
print("  · macro-F1 도 배율하락도 그대로                     → 증강 부족. hard 프리셋 또는 f320")
print("  · macro-F1 이 크게 떨어졌다                          → 증강이 병변을 잘라먹는 중.")
print("                                                        더 여유 있는 크롭이 필요합니다")

## 8. 크롭 방식 비교 실험

이제 **어떤 크롭이 제일 좋은지** 정합니다. 2단계(병변 6종)로 비교합니다 —
크롭이 영향을 주는 건 "병변의 형태를 구분하는" 쪽이기 때문입니다.

> ⚠️ **배포 관점의 함정**: `m1.5`/`m2.5` 크롭은 **정답 박스를 알고 있다는 전제**입니다.
> 실제 보호자 사진에는 박스가 없습니다. 그래서
> - `m1.5` / `m2.5` 점수 = 병변 위치를 아는 경우의 **상한선**
> - `full` 점수 = 실제 서비스에 가까운 **정직한 숫자**
>
> 최종 배포는 (a) `full` 로 학습하거나 (b) 병변 검출 모델을 앞에 붙여야 합니다.
> 지금은 둘 다 재서 격차를 확인합니다.

In [ ]:
import gc

crop_results = {}
quick = CFG(model_name="tf_efficientnetv2_s.in21k_ft_in1k", img_size=224,
            epochs=6, balance_strategy="class_weight", monitor="macro_f1")

for tag in crop.available_tags():
    print(f"\n{'='*60}\n  크롭: {tag}\n{'='*60}")
    d_tag = crop.switch_tag(df, tag)                  # 같은 매니페스트, 크롭만 교체
    s2_tag = stages.to_stage2(d_tag, verbose=False)   # 2단계로 비교
    t, v = split.get_fold(s2_tag, 0)
    m = models.build("effnetv2_s", len(CLASSES), pretrained=True, verbose=False)
    ltr, lva, dtr, _ = data.build_loaders(t, v, quick, model=m, classes=CLASSES)
    train.fit(m, ltr, lva, CFG(**{**quick.to_dict(), "exp_name": f"crop_{tag}"}),
              ds_train=dtr, verbose=True)
    _, lg, yy = train.evaluate_loader(m, lva, None, DEV, len(CLASSES))
    crop_results[tag] = evaluate.full_report(lg, yy, CLASSES, show=False)
    del m; gc.collect()
    if DEV == "cuda":
        torch.cuda.empty_cache()

evaluate.compare_models(crop_results)

In [ ]:
BEST_CROP = max(crop_results, key=lambda k: crop_results[k].metrics["macro_f1"])
print(f"\n✅ 2단계 최적 크롭: {BEST_CROP}")
if "full" in crop_results and BEST_CROP != "full":
    gap = (crop_results[BEST_CROP].metrics["macro_f1"]
           - crop_results["full"].metrics["macro_f1"])
    print(f"   full 대비 격차: +{gap:.4f}")
    print("   ⚠️ 이 격차가 '병변 위치를 안다'는 정보의 값입니다.")
    print("      실제 서비스에서는 그 정보가 없으니 full 점수를 기준으로 기대치를 잡으세요.")

# 다음 노트북들이 읽어갑니다
import json
(env.work_root()/"best_crop.txt").write_text(BEST_CROP)
(env.work_root()/"stage1_threshold.json").write_text(json.dumps({
    "threshold": THR1, "target_recall": cfg1.target_recall_stage1,
    "auroc": bin1["auroc"], "precision_at_target": bin1["precision_at_target"],
    "stage1_crop": STAGE1_CROP,        # ← 1단계는 다른 크롭일 수 있습니다
    "stage2_crop": BEST_CROP,
    "audit": {k: v for k, v in report.items() if not isinstance(v, dict)},
}, indent=2, ensure_ascii=False))
print("\n저장: best_crop.txt, stage1_threshold.json")
print(f"  1단계 크롭 {STAGE1_CROP} / 2단계 크롭 {BEST_CROP}")

---
## ✅ 정리

| 확인한 것 | 어디서 | 통과 기준 |
|---|---|---|
| 크롭이 배율로 정답을 흘리지 않는가 | 1번 감사 `[1]` | 정상/병변 1.5배 미만, 병변끼리 2배 미만 |
| `full` 로 바꿔도 병변을 안 잃는가 | 1번 `full_crop_loss` | 천장 ≥ 0.98 |
| 사진 없이 얼마나 맞히는가 (하한선) | 1번 `shortcut_baseline` | 2단계 < 0.30 |
| 1단계가 정상/이상을 구분한다 | 3번 | AUROC > 0.80, 하한선 대비 +0.05 |
| 2단계가 랜덤보다 낫다 | 4번 | macro-F1 > 0.25, 하한선 대비 +0.15 |
| **이어붙인 실제 성능** | 5번 | 스크리닝 recall ≥ 0.95 |
| **실사용에서 버티는가** | 6번 | 배율·위치 하락 < 15% |
| 증강만으로 해결되는가 | 7번 | 하락폭이 줄면 재크롭 불필요 |
| 쓸 크롭 | 8번 | — |

## ✅ 다음 단계

`04_학습_최신모델_비교.ipynb` — 2단계 모델을 최신 백본들로 바꿔가며 비교합니다.

단, **6번 하락폭이 30%를 넘었다면 먼저 크롭을 고치세요.** 지름길이 열린 상태에서
백본을 비교하면 "배율을 가장 잘 읽는 모델"을 뽑게 됩니다.

📖 함께 읽기: [`docs/basics/09_ViT와_최신_백본_지도_2026.md`](../docs/basics/09_ViT와_최신_백본_지도_2026.md),
[`docs/cautions/08_2단계_파이프라인_설계_주의점.md`](../docs/cautions/08_2단계_파이프라인_설계_주의점.md)